In [13]:

!pip install opencv-python


  Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl (44.0 MB)
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.4 MB 2.1 MB/s eta 0:00:06
   -- ------------------------------------- 0.8/12.4 MB 1.5 MB/s eta 0:00:08
   --- ------------------------------------ 1.0/12.4 MB 1.4 MB/s eta 0:00:09
   --- ------------------------------------ 1.0/12.4 MB 1.4 MB/s eta 0:00:09
   --- ------------------------------------ 1.0/12.4 MB 1.4 MB/s eta 0:00:09
   --- ------------------------------------ 1.0/12.4 MB 1.4 MB/s eta 0:00:09
   --- ------------------------------------ 1.0/12.4 MB 1.4 MB/s eta 0:00:09
   ----- ---------------------------------- 1.6/12.4 MB 822.3 kB/s eta 0:00:14
   ------- ------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.5.1 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.5.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.
scipy 1.13.1 requires numpy<2.3,>=1.22.4, but you have numpy 2.5.1 which is incompatible.


In [ ]:
import cv2
import numpy
import yt_dlp


In [ ]:


# Access the camera feed (0 for default webcam)
cap = cv2.VideoCapture(0)

frames = []
gap = 5  # Difference between current frame and the 5th previous frame [4]
count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convert frame to grayscale for processing [5]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    frames.append(gray)

    # Maintain a sliding window of frames based on the gap [6, 7]
    if len(frames) > (gap + 1):
        frames.pop(0)

    # Begin detection logic once the buffer is full [2]
    if len(frames) == (gap + 1):
        # Calculate absolute difference between the first and last frame in buffer [2, 8]
        diff = cv2.absdiff(frames, frames[-1])
        
        # Apply thresholding to ignore minor pixel flickers [8]
        _, thresh = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)
        
        # Find contours (clusters of moving pixels) [9, 10]
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        motion_detected = False
        for c in contours:
            # Only consider contours with a significant area (e.g., > 500 pixels) [3, 11]
            if cv2.contourArea(c) < 500:
                continue
            
            motion_detected = True
            # Draw a green bounding rectangle around the detected motion [3, 12]
            (x, y, w, h) = cv2.boundingRect(c)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        if motion_detected:
            cv2.putText(frame, "MOTION DETECTED", (10, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2) [13]

    # Display the result [14]
    cv2.imshow("Burglar Alarm System", frame)
    
    # Exit on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

SyntaxError: 'return' outside function (3105403344.py, line 14)

In [ ]:
# Example usage (Replace with your chosen YouTube link)

get_youtube_cap_segment(video_url, start_second=10, duration_seconds=2)

In [18]:


def get_youtube_cap_segment(youtube_url, start_second=0, duration_seconds=2):
    # 1. Configure yt-dlp to extract the streaming URL
    ydl_opts = {
        'format': 'best[ext=mp4]/best',  # Fetch the best compatible MP4 stream
        'quiet': True
    }
    
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=False)
        stream_url = info['url']
    
    # 2. Pass the direct stream URL into cv2.VideoCapture
    cap = cv2.VideoCapture(stream_url)
    if not cap.isOpened():
        print("Error: Could not open YouTube stream.")
        return None
        # Access the camera feed (0 for default webcam)
    cap = cv2.VideoCapture(0)

    frames = []
    gap = 5  # Difference between current frame and the 5th previous frame [4]
    count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Convert frame to grayscale for processing [5]
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frames.append(gray)

        # Maintain a sliding window of frames based on the gap [6, 7]
        if len(frames) > (gap + 1):
            frames.pop(0)

        # Begin detection logic once the buffer is full [2]
        if len(frames) == (gap + 1):
            # Calculate absolute difference between the first and last frame in buffer [2, 8]
            diff = cv2.absdiff(frames, frames[-1])
            
            # Apply thresholding to ignore minor pixel flickers [8]
            _, thresh = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)
            
            # Find contours (clusters of moving pixels) [9, 10]
            contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            motion_detected = False
            for c in contours:
                # Only consider contours with a significant area (e.g., > 500 pixels) [3, 11]
                if cv2.contourArea(c) < 500:
                    continue
                
                motion_detected = True
                # Draw a green bounding rectangle around the detected motion [3, 12]
                (x, y, w, h) = cv2.boundingRect(c)
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

            if motion_detected:
                cv2.putText(frame, "MOTION DETECTED", (10, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2) [13]

        # Display the result [14]
        cv2.imshow("Burglar Alarm System", frame)
        
        # Exit on 'q' key press
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        # Clean up resources
        cap.release()
        cv2.destroyAllWindows()

    # Example usage (Replace with your chosen YouTube link)
    video_url = "https://www.youtube.com/watch?v=8FG-wBXs0Es&t=3454s"
    get_youtube_cap_segment(video_url, start_second=10, duration_seconds=10)


In [11]:


def get_youtube_cap_segment(youtube_url, start_second=0, duration_seconds=2):
    # 1. Configure yt-dlp to extract the streaming URL
    ydl_opts = {
        'format': 'best[ext=mp4]/best',  # Fetch the best compatible MP4 stream
        'quiet': True
    }
    
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=False)
        stream_url = info['url']
    
    # 2. Pass the direct stream URL into cv2.VideoCapture
    cap = cv2.VideoCapture(stream_url)
    if not cap.isOpened():
        print("Error: Could not open YouTube stream.")
        return None

    # 3. Calculate frames and jump to the start timestamp
    fps = cap.get(cv2.CAP_PROP_FPS)
    start_frame = int(start_second * fps)
    total_frames_to_read = int(duration_seconds * fps)
    
    # Set the video capture pointer to the starting frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    print(f"Streaming {duration_seconds} seconds starting from {start_second}s...")
    print(f"Reading exactly {total_frames_to_read} frames at {fps} FPS.")
    
    frames_read = 0
    while cap.isOpened() and frames_read < total_frames_to_read:
        ret, frame = cap.read()
        if not ret:
            break
        
        # --- Process your 2-second clip frame here ---
        cv2.imshow('YouTube 2-Sec Clip', frame)
        frames_read += 1
        
        # Break loop if 'q' is pressed
        if cv2.waitKey(int(1000 / fps)) & 0xFF == ord('q'):
            break
            
    # Clean up resources
    cap.release()
    cv2.destroyAllWindows()

# Example usage (Replace with your chosen YouTube link)
video_url = "https://www.youtube.com/watch?v=8FG-wBXs0Es&t=3454s"
get_youtube_cap_segment(video_url, start_second=10, duration_seconds=10)


Streaming 10 seconds starting from 10s...
Reading exactly 250 frames at 25.0 FPS.
